In [117]:
from langchain.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.schema import Document
import json
import os
from dotenv import load_dotenv
from concurrent.futures import ThreadPoolExecutor

import glob
import chromadb
from langchain_core.runnables import RunnableLambda
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import List
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler


In [83]:
#Embeddings
load_dotenv()
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [84]:
data_dir = os.getenv("DATA_DIR")
vectordb_dir = os.getenv("VECTORDB_DIR", "./chroma")
EMBEDDING_MODEL_NAME = os.getenv("EMBEDDING_MODEL", "all-MiniLM-L6-v2")
LLM_PROVIDER = os.getenv("LLM_PROVIDER", "gemini")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")


In [85]:
print(data_dir)

./Data/


In [86]:
def index_doc_in_chroma(data_dir, vectordb_dir):
    for filename in os.listdir(data_dir):
        
        if filename.endswith(".json"):
            #print(f"Reading { filename} \n")
            filepath = os.path.join(data_dir, filename)
            collection_name = os.path.splitext(filename)[0]
            #loading the json
            with open(filepath, "r", encoding="utf-8") as f:
                try:
                    json_data = json.load(f)
                    # print(f"json_data: {json_data[0]}")
                    # print("\n")
                except Exception as e:
                    print(f"Skipping {filename}: Error: {e}")
                    continue
            #ensuring its a list of obj
            if not isinstance(json_data, list):
                print(f"Skipping {filename}: JSON is not a list of objects.")
                continue

            documents = []
            for idx, obj in enumerate(json_data):
                text = json.dumps(obj, indent=2)
                metadata = {
                    "source_file": filename,
                    "index": idx,
                    "keys": ",".join(obj.keys())
                }
                documents.append(Document(page_content=text, metadata= metadata))

            collection_dir = os.path.join(vectordb_dir, collection_name)
            vector_store = Chroma.from_documents(
                documents=documents,
                embedding= embedding_model,
                collection_name= collection_name,
                persist_directory= collection_dir
            )

            vector_store.persist()
            print(f"Indexed {len(documents)} docs into collection '{collection_name}'")            


In [87]:
index_doc_in_chroma(data_dir, vectordb_dir)

/var/folders/j7/ytxcvpvj7sg85_k31gzfzg5c0000gn/T/ipykernel_99226/1175727996.py:40: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vector_store.persist()


Indexed 373 docs into collection 'AllFacultyGeneralInformation'
Indexed 2487 docs into collection 'AllCourseRelatedData'
Indexed 373 docs into collection 'AllFacultyResearchInformation'


In [88]:
#loading the data
vector_store = Chroma(
    collection_name="AllCourseRelatedData",
    persist_directory="./VectorDB/AllCourseRelatedData",
    embedding_function=embedding_model
)

results = vector_store.similarity_search("NLP",k=2)
for doc in results:
    print(doc.page_content)
    print(doc.metadata)

{
  "Course Code": "CS 584",
  "Course Title": "Natural Language Processing",
  "Course URL": "https://stevens.smartcatalogiq.com/en/2024-2025/academic-catalog/courses/cs-computer-science/500/cs-584",
  "Course Name": "Natural Language Processing",
  "Course Number": 584.0,
  "Course Description": "Natural language processing (NLP) is one of the most important technologies in the era of information. Comprehending human language is also a crucial and challenging part of artificial intelligence. People communicate almost everything in language: conferences, emails, customer service, language translation, web searches, reports, etc. There are a large variety of underlying tasks and machine learning models behind NLP applications. Recently, deep learning approaches have achieved high performance in many different NLP tasks. Instead of traditional and task-specific feature engineering, deep learning can solve tasks with single end-to-end models. The course provides an introduction to machin

In [89]:
#loading the data
vector_store = Chroma(
    collection_name="AllFacultyGeneralInformation",
    persist_directory="./VectorDB/AllFacultyGeneralInformation",
    embedding_function=embedding_model
)

results = vector_store.similarity_search("TIAN Han",k=2)
for doc in results:
    print(doc.page_content)
    print(doc.metadata)

{
  "FacultyGeneralInfo": "Yi Guo Academics https://www.stevens.edu/profile/yguo1 Burchard 202 (201) 216-5658 [email\u00a0protected] Website PhD (1999) University of Sydney, Australia (Electrical and Information Engineering) MS (1995) Xi\u2019an University of Technology, China (Electrical Engineering) BS (1992) Xi\u2019an University of Technology, China (Electrical Engineering) Yi Guo joined Department of Electrical and Computer Engineering at Stevens Institute of Technology in 2005, and is currently Thomas E. Hattrick Chair Professor. Before that, she was Visiting Assistant Professor in ECE Department of University of Central Florida between 2002 and 2005. Dr. Guo received her Ph.D degree in Electrical and Information Engineering in 1999 from University of Sydney, Australia. After her Ph.D, she worked as a postdoctoral research fellow in Computer Science and Mathematics Division at Oak Ridge National Laboratory between 2000 and 2002. Stevens Institute of Technology, Department of Elec

In [90]:
#loading the data
vector_store = Chroma(
    collection_name="AllFacultyResearchInformation",
    persist_directory="./VectorDB/AllFacultyResearchInformation",
    embedding_function=embedding_model
)

results = vector_store.similarity_search("Mixture of Experts",k=2)
for doc in results:
    print(doc.page_content)
    print(doc.metadata)

{
  "FacultyResearchInfo": "Tian Han 54 Unsupervised/Semi-supervised Learning, Probabilistic Generative Modeling, Explainable AI, Computer Vision. CISE-RI: NSF CAREER Award (2024) "
}
{'keys': 'FacultyResearchInfo', 'source_file': 'AllFacultyResearchInformation.json', 'index': 178}
{
  "FacultyResearchInfo": "Tian Han 54 Unsupervised/Semi-supervised Learning, Probabilistic Generative Modeling, Explainable AI, Computer Vision. CISE-RI: NSF CAREER Award (2024) "
}
{'source_file': 'AllFacultyResearchInformation.json', 'keys': 'FacultyResearchInfo', 'index': 178}


## Retrieval

In [91]:
#loading all collections from directory
def load_all_vectorized_collection(path="./VectorDB/"):
    collections={}
    for folder in os.listdir(path):
        full_path = os.path.join(path, folder)
        if os.path.isdir(full_path):
            collections[folder] = Chroma(
                collection_name= folder,
                persist_directory= full_path,
                embedding_function= embedding_model
            )
    return collections

In [92]:
print(load_all_vectorized_collection())

{'AllFacultyResearchInformation': <langchain_community.vectorstores.chroma.Chroma object at 0x36e14a0f0>, 'AllFacultyGeneralInformation': <langchain_community.vectorstores.chroma.Chroma object at 0x36e1495e0>, 'AllCourseRelatedData': <langchain_community.vectorstores.chroma.Chroma object at 0x36e149850>}


In [93]:
def dynamic_retriever(query, k=10, top_n_collections= 2 ):
    collections = load_all_vectorized_collection()
    all_docs = []
    #parallel search
    def search_collection(name, store):
        docs = store.similarity_search_with_score(query, k=4)
        for doc, score in docs:
            doc.metadata["collection"] = name
            doc.metadata["similarity_score"] = score
            all_docs.append(doc)
    
    with ThreadPoolExecutor() as executer:
        futures = [executer.submit(search_collection, name, store)
                   for name, store in collections.items()
                   ]
        
        for future in futures:
            
            try:
                docs = future.result()
                all_docs.extend(docs)
            except Exception as e: 
                print(f"Error during retrieval: {e}")
    #filtering duplicates
    seen_ids = set()
    unique_ranked_docs = []
    #sorting by similarity score (lower is better)
    print(all_docs)
    all_docs.sort(key=lambda d:  d.metadata.get("similarity_score", 1e6))

    for doc in all_docs:
        doc_id = doc.metadata.get("id", doc.page_content)
        if doc_id not in seen_ids:
            unique_ranked_docs.append(doc)
            seen_ids.add(doc_id)
        if len(unique_ranked_docs) >=k:
            break

    return unique_ranked_docs
    

            

In [94]:
user_qery = "All researches Mixture of Expert?"
docs =dynamic_retriever(query=user_qery, k=10)
print("\n Top results: ")
for doc in docs:
    print(f"[{doc.metadata['collection']}] {doc.page_content[:100]}...")

Error during retrieval: 'NoneType' object is not iterable
Error during retrieval: 'NoneType' object is not iterable
Error during retrieval: 'NoneType' object is not iterable
[Document(metadata={'index': 185, 'keys': 'FacultyResearchInfo', 'source_file': 'AllFacultyResearchInformation.json', 'collection': 'AllFacultyResearchInformation', 'similarity_score': 1.162785530090332}, page_content='{\n  "FacultyResearchInfo": "Jonggi Hong 61 My research lies in tackling real-world challenges through the convergence of human-computer interaction and cutting-edge machine-learning methodologies. My work revolves around the exploration of user-system interactions, employing both quantitative and qualitative analysis techniques to gain deeper insights into the dynamics between individuals and intelligent systems.  "\n}'), Document(metadata={'index': 185, 'keys': 'FacultyResearchInfo', 'source_file': 'AllFacultyResearchInformation.json', 'collection': 'AllFacultyResearchInformation', 'similarity_scor

### BUilding different Retrival approach

In [115]:
#Defining LLM
def get_llm():
    if LLM_PROVIDER=="gemini":
        return GoogleGenerativeAI(model="models/gemini-pro", google_api_key=GOOGLE_API_KEY)
    elif LLM_PROVIDER=="openai":
        return ChatOpenAI(model="gpt-4", openai_api_key=OPENAI_API_KEY)
    else:
        return ValueError("Unsupported LLM Provider")

In [96]:
def load_vector_store(collection_name: str) -> Chroma:
    collection_path = os.path.join(vectordb_dir, collection_name)
    return Chroma(
        collection_name=collection_name,
        persist_directory=collection_path,
        embedding_function=embedding_model
    )

In [97]:
def get_all_collections() -> List[str]:
    return [
        name for name in os.listdir(vectordb_dir)
        if os.path.isdir(os.path.join(vectordb_dir, name))
    ]


In [98]:
def ask_router_llm(llm, user_query: str, collections: List[str]) -> List[str]:
    prompt = f"""
                You are a smart router in a RAG system.

                Your task is to choose the most relevant collections from this list:
                {collections}

                User question:
                "{user_query}"

                Return ONLY a valid JSON array of collection names. No explanation.
            """
    response = llm.invoke(prompt)
    try:
        return json.loads(response.content.strip())
    except Exception:
        print("Could not parse LLM output. Falling back to all collections.")
        return collections

In [99]:
def retrieve_docs_from_collections(user_query: str, collection_names: List[str], top_k: int = 5) -> List[Document]:
    all_docs = []
    for name in collection_names:
        vs = load_vector_store(name)
        docs = vs.similarity_search(user_query, k=top_k)
        all_docs.extend(docs)
    return all_docs

In [100]:
def build_prompt(context_docs: List[Document], user_query: str, chat_summary: str = None, recent_chats: List[str] = []) -> str:
    context = "\n\n".join([doc.page_content for doc in context_docs])
    recent = "\n".join(recent_chats[-3:]) if recent_chats else ""
    summary = chat_summary or ""

    return f"""
You are a university assistant bot.

Relevant document snippets:
{context}

Chat Summary:
{summary}

Recent Conversation:
{recent}

Answer the user question below. If the information is insufficient, respond with: "I need to do a web search."

User Query:
{user_query}
"""

In [101]:
def final_answer_llm(llm, prompt: str) -> str:
    return llm.invoke(prompt).content


In [102]:
def process_user_query(user_query: str, chat_summary: str = None, recent_chats: List[str] = []) -> str:
    llm = get_llm()
    collections = get_all_collections()
    selected = ask_router_llm(llm, user_query, collections)
    retrieved_docs = retrieve_docs_from_collections(user_query, selected)
    print(retrieved_docs)
    final_prompt = build_prompt(retrieved_docs, user_query, chat_summary, recent_chats)
    return final_answer_llm(llm, final_prompt)

In [106]:
query = "Can you give me some research on AI topics ?"
print(process_user_query(query))

[Document(metadata={'keys': 'FacultyResearchInfo', 'index': 354, 'source_file': 'AllFacultyResearchInformation.json'}, page_content='{\n  "FacultyResearchInfo": "Zining Zhu 230 I direct the Explainable and Controllable AI Lab, where we research the foundations and application of approaches that make AI explainable and controllable. The areas of research include: - Model interpretability - Natural language explanation - Model intervention - Societal implications and safe deployments - Outstanding Paper Award, NAACL 2025 - Top Reviewer Award, NeurIPS 2023 "\n}'), Document(metadata={'source_file': 'AllFacultyResearchInformation.json', 'index': 354, 'keys': 'FacultyResearchInfo'}, page_content='{\n  "FacultyResearchInfo": "Zining Zhu 230 I direct the Explainable and Controllable AI Lab, where we research the foundations and application of approaches that make AI explainable and controllable. The areas of research include: - Model interpretability - Natural language explanation - Model inte

In [52]:
get_all_collections()

['AllFacultyResearchInformation',
 'AllFacultyGeneralInformation',
 'AllCourseRelatedData']

In [107]:
def parse_resume_text_auto_schema(resume_text: str) -> dict:
    llm = get_llm()

    prompt = f"""
        You are a professional resume parser.

        Extract all structured and meaningful information from the resume text below and return it as a well-formatted JSON object.

        Group information into appropriate fields like personal details, skills, education, experience, projects, certifications, social links, etc.

        Do not include raw text blocks. Only return structured JSON.

        Resume Text:
        \"\"\"
        {resume_text}
        \"\"\"
     """

    response = llm.invoke(prompt)
    try:
        return json.loads(response.content.strip())
    except Exception as e:
        print("Failed to parse LLM response:", e)
        print("Raw Output:", response.content)
        return {"error": "Invalid JSON from LLM"}


In [110]:
resume_text ='''Nitin Sunil Chaube 
nitinchaube08@gmail.com | +1 2019187533 | Jersey City, NJ | GitHub | LinkedIn
EDUCATION
•	MS in Computer Science	New Jersey, US
Stevens Institute of Technology	Sept 2024 May 2026
•	Bachelor’s in computer science	Mumbai, IND
Mumbai University	Aug 2018  May 2022
SKILLS		
•	Machine Learning: Regression, Classification, Clustering, Transformers, NLP, LLM, Agentic AI, RLHF, Recommendations Systems, ML Pipelines, Model de-ployment, A/B Testing, GPT, OpenAI, RAG, Fine Tuning, Transfer Learning, Gym, Distributed Systems, Parallel Programming, High Performance Computing
•	Python Programming and Libraries: NumPy, Pandas, Sci-kit Learn, TensorFlow, Pytorch, NLTK, Matplotlib, OpenCV, Chroma DB, Lang Chain.
•	Languages and Databases: Python, C++, Java, SQL, PySpark, MongoDB, Reactjs, Nodejs, JavaScript.
•	Others: Git, Unix, Docker, AWS, GCP, Flask, MongoDB, Splunk, Dynatrace, Power BI, Tableau.
Experience
Stevens Institute of Technology | Teaching Assistant	Jan 2025 May 2025
•	Led weekly lab sessions for graduate-level Data Structures, guiding students through complex coding exercises and practical applications of fundamental algo-rithms like sorting, searching, graph traversals, etc.
•	Provided in-depth, individualized support to students, expertly debugging code, clarifying challenging concepts (e.g., Recursion, Bit manipulation, tree struc-tures, hash tables), and actively fostering independent data structure skills.
•	Collaborated directly with the professor on assignment development, comprehensive grading of 80% of assignments, and course material management, ensur-ing accurate student progress tracking and constructive feedback effectively.
TIAA | Software Developer 	July 2022 Aug 2024
•	Architected and deployed Python-based automation solutions leveraging deep learning (BERT, NLP) for Proof-of-Concepts that optimized operational workflows, reducing manual intervention and boosting efficiency by 70%. Partnered with DevOps, Security, and Database teams to troubleshoot complex incidents, and refine monitoring techniques, ensuring seamless system integration and issue prevention.
•	Maintained 99%+ system uptime through regular monitoring with Splunk and Dynatrace for real-time log analysis and anomaly detection, leading to the diag-nosis and resolution of over 500 critical system issues and significant performance enhancement.
•	Led the development of an AI-driven incident management tool, training Bi-LSTM and BERT models on 30,000+ historical incidents to achieve 94%+ classifica-tion accuracy which automated 80% of issue assignments.
•	Engineered and deployed NLP-based incident detection and classification algorithms, eliminating reliance on manual triaging and significantly accelerating inci-dent response times through streamlined issue classification and assignment, drastically saving over 100+ operational hours.

Cloud Counselage | AI Intern 	Apr 2021 – Sept 2021
•	Spearheaded a team of 15+ junior interns across diverse projects in Machine Learning and Natural Language Processing, driving timely project completion through the strategic implementation of Agile methodologies and sprint planning.
•	Mentored and guided the team on 3 projects involving model development, providing comprehensive training in feature engineering, time-series modelling, and complex NLP pipelines, which significantly elevated their technical proficiency and project contributions.
Projects
Advisor AI (LLM Powered Academic Advisor System)  (Github)
•	Implementing a full-stack chat bot using React, Flask, and Lang Chain to provide 98% intelligent student support. This includes a Retrieval-Augmented Genera-tion (RAG) architecture, leveraging a ChromaDB vector store for semantic search across college course, professor, and general information.
•	Seamless integration of both OpenAI APIs and local open-source LLMs (Llama) for flexible response generation, reducing API costs by 30%. Build a robust con-versational memory system using MongoDB to persist chat history and maintain context across user sessions.
•	Integrating a user feedback loop (thumbs up/down) with data stored in MongoDB, enabling a Reinforcement Learning from Human Feedback (RLHF) mechanism. This system is for continuous improvement of bot responses, aiming to increase answer satisfaction by 15% over time.
•	Optimizing information retrieval with ChromaDB and advanced embedding models (Sentence Transformers) to achieve over 90% high-accuracy semantic re-trieval of relevant information, ensuring grounded and contextually appropriate answers to student inquiries.
DIG (Disaster Information Graph)  (International Research Journal, ISSN)
•	Conducted extensive research on CrisisLex and CrisisNLP datasets, consolidating over 500,000 crisis-related tweets to develop an AI-driven model for accu-rate crisis classification and enhanced disaster response insights by extracting the valuable information from tweets.
•	Trained and evaluated CNN, BiLSTM, and BERT models, achieving 87%+ accuracy in binary classification and 82%+ in multiclass classification of informative-ness and other crisis-related categories, improving categorization of tweets into disaster related events.
•	Integrated OpenAI LLM to factually extract critical information from crisis-related tweets and developed a user-friendly UI to graphically represent the extract-ed data, leading to a 25% improvement in enhancing data interpretability for rapid insights during disasters.
KeosWorld (E-commerce webpage) (Github)
•	A Comprehensive Full-Stack E-commerce end-to-end web application supporting over 500 products using React, Redux, and design frameworks for a frontend, integrated with a scalable Node.js/Express backend and MongoDB Atlas for efficient cloud database management.
•	 Firebase for robust email authentication and Stripe for secure, efficient payment processing. This system supports diverse product variants and provides real-time inventory updates at checkout, ensuring 99% accurate order fulfillment.
•	Implemented user-facing features such as product Browse, shopping cart management, and streamlined order placement. Concurrently, developed a powerful admin dashboard that improved product, category, and inventory management by over 30%, facilitating rapid updates.
Achievements and Certifications
•	Led a 15-member ROBOCON team as Vice-Captain to secure an All-India Rank 9 in the prestigious ABU ROBOCON competition.
•	Artificial Intelligence Engineer by Simplilearn  | Competitive Programming by Coding Ninjas	'''


In [111]:
resp = parse_resume_text_auto_schema(resume_text=resume_text)

In [112]:
resp

{'Personal Details': {'Name': 'Nitin Sunil Chaube',
  'Email': 'nitinchaube08@gmail.com',
  'Phone': '+1 2019187533',
  'Location': 'Jersey City, NJ',
  'Social Links': {'GitHub': 'GitHub', 'LinkedIn': 'LinkedIn'}},
 'Education': [{'Degree': 'MS in Computer Science',
   'Institution': 'Stevens Institute of Technology',
   'Location': 'New Jersey, US',
   'Start Date': 'Sept 2024',
   'End Date': 'May 2026'},
  {'Degree': 'Bachelor’s in computer science',
   'Institution': 'Mumbai University',
   'Location': 'Mumbai, IND',
   'Start Date': 'Aug 2018',
   'End Date': 'May 2022'}],
 'Skills': ['Machine Learning',
  'Python Programming',
  'Python Libraries',
  'Languages and Databases',
  'Git',
  'Unix',
  'Docker',
  'AWS',
  'GCP',
  'Flask',
  'MongoDB',
  'Splunk',
  'Dynatrace',
  'Power BI',
  'Tableau'],
 'Experience': [{'Company': 'Stevens Institute of Technology',
   'Position': 'Teaching Assistant',
   'Start Date': 'Jan 2025',
   'End Date': 'May 2025',
   'Responsibilities': 

In [118]:
## For Streaming the tokens on web we can do below:
from langchain.callbacks.base import BaseCallbackHandler
from langchain_google_genai import ChatGoogleGenerativeAI
class TokenStreamHandler(BaseCallbackHandler):
    def __init__(self):
        self.tokens=[]
    
    def on_llm_new_token(self, token, **kwargs):
        self.tokens.append(token)

    def get_stream(self):
        for token in self.tokens:
            yield token


def get_llm(handler):
    if LLM_PROVIDER == "gemini":
        return ChatGoogleGenerativeAI(
            model="gemini-pro",
            google_api_key=GOOGLE_API_KEY,
            streaming=True,
            callbacks=[handler]
        )
    elif LLM_PROVIDER == "openai":
        return ChatOpenAI(
            model="gpt-4",
            openai_api_key=OPENAI_API_KEY,
            streaming=True,
            callbacks=[handler]
        )
    else:
        raise ValueError("Unsupported provider")
    


In [119]:
def process_query(user_query: str) -> str:
    handler = TokenStreamHandler()
    llm = get_llm(handler)

    # Build your prompt as needed
    prompt = f"Answer the following: {user_query}"

    # This triggers streaming callbacks
    _ = llm.invoke(prompt)

    # Yield streamed tokens
    return handler.get_stream()

In [121]:
query = "what is quantum computing?"
stream = process_query(query)
for token in stream:
    print(token)


Quant
um
 computing
 is
 a
 type
 of
 computation
 that
 uses
 quantum
 bits
,
 or
 '
q
ubits
',
 which
 can
 exist
 in
 multiple
 states
 at
 once
,
 as
 opposed
 to
 traditional
 computing
 binary
 bits
 that
 exist
 in
 a
 state
 of
 either
 '
0
'
 or
 '
1
'.
 This
 allows
 quantum
 computers
 to
 process
 vast
 amounts
 of
 information
 and
 perform
 complex
 calculations
 much
 faster
 than
 classical
 computers
.
 Quantum
 computers
 utilize
 the
 principles
 of
 quantum
 mechanics
,
 including
 super
position
 and
 ent
ang
lement
,
 to
 process
 information
.



In [ ]:
@app.route("/chat", methods=["POST"])
def chat():
    data = request.get_json()
    query = data.get("query", "")
    stream = process_query(query)
    return Response(stream_with_context(stream), mimetype="text/plain")

if __name__ == "__main__":
    app.run(debug=True)
